In [0]:
%sql

CREATE TABLE IF NOT EXISTS mba.trusted.f_bandeira
(
    MesCompetencia INT COMMENT 'Mês de competência no formato YYYYMM',
    IsVermelha TINYINT COMMENT 'Indicador de bandeira vermelha: 1 = bandeira vermelha acionada; 0 = demais bandeiras'
)
USING DELTA

COMMENT 'Dimensão mensal da bandeira tarifária de energia elétrica';

In [0]:
%sql

MERGE INTO mba.trusted.f_bandeira AS tgt
USING (
    SELECT
        CAST(
            date_format(DatCompetencia, 'yyyyMM')
            AS INT
        ) AS MesCompetencia,

        MAX(
            CASE
                WHEN NomBandeiraAcionada LIKE 'Vermelha%' THEN 1
                ELSE 0
            END
        ) AS IsVermelha

    FROM mba.raw.bandeira_acionada

    WHERE DatCompetencia IS NOT NULL

    GROUP BY
        date_format(DatCompetencia, 'yyyyMM')

) AS src

ON tgt.MesCompetencia = src.MesCompetencia

WHEN NOT MATCHED THEN

    INSERT
    (
        MesCompetencia,
        IsVermelha
    )

    VALUES
    (
        src.MesCompetencia,
        src.IsVermelha
    );